<a href="https://colab.research.google.com/github/kyleonyonyi-a11y/AI/blob/main/stroke_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

P4 Gathering and preparing appropriate datasets for an AI solution

Download and load the data

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [9]:
from google.colab import files
uploaded = files.upload()   # choose healthcare-dataset-stroke-data.csv
path = "."

Saving healthcare-dataset-stroke-data.csv to healthcare-dataset-stroke-data (1).csv


Audit the data

In [12]:
import pandas as pd

# Load the dataset before auditing
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

df.info()
print(df.isnull().sum())
print('Duplicates:', df.duplicated().sum())
print(df['stroke'].value_counts())
print(df['stroke'].value_counts(normalize=True).round(4))
print(df['gender'].value_counts())
print(df['smoking_status'].value_counts())
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_ty

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


Clean the data

In [13]:
import pandas as pd

path = "."
df = pd.read_csv(f"{path}/healthcare-dataset-stroke-data.csv")
df_clean = df.drop(columns=['id'])
df_clean = df_clean[df_clean['gender'] != 'Other'].reset_index(drop=True)
print(df_clean.shape)

(5109, 11)


Encode categorical variables

In [19]:
df_clean['gender'] = df_clean['gender'].map({'Male': 0, 'Female': 1})
df_clean['ever_married'] = df_clean['ever_married'].map({'No': 0, 'Yes': 1})
df_clean['Residence_type'] = df_clean['Residence_type'].map({'Rural': 0, 'Urban': 1})

df_enc = pd.get_dummies(df_clean, columns=['work_type', 'smoking_status'],
                        drop_first=True, dtype=int)
print(df_enc.shape)
df_enc.head()

(5109, 16)


,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,stroke,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,NaN,67.0,0,1,NaN,NaN,228.69,36.6,1,0,1,0,0,1,0,0
1,NaN,61.0,0,0,NaN,NaN,202.21,NaN,1,0,0,1,0,0,1,0
2,NaN,80.0,0,1,NaN,NaN,105.92,32.5,1,0,1,0,0,0,1,0
3,NaN,49.0,0,0,NaN,NaN,171.23,34.4,1,0,1,0,0,0,0,1
4,NaN,79.0,1,0,NaN,NaN,174.12,24.0,1,0,0,1,0,0,1,0


split train and testing data

In [15]:
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Load the original dataset
path = "."
df = pd.read_csv(f"{path}/healthcare-dataset-stroke-data.csv")

# 2. Re-apply cleaning
df_clean = df.drop(columns=['id'])
df_clean = df_clean[df_clean['gender'] != 'Other'].reset_index(drop=True)

# 3. Apply encoding
df_clean['gender'] = df_clean['gender'].map({'Male': 0, 'Female': 1})
df_clean['ever_married'] = df_clean['ever_married'].map({'No': 0, 'Yes': 1})
df_clean['Residence_type'] = df_clean['Residence_type'].map({'Rural': 0, 'Urban': 1})

df_enc = pd.get_dummies(df_clean, columns=['work_type', 'smoking_status'], drop_first=True, dtype=int)

# 4. Split dataset
X = df_enc.drop(columns='stroke')
y = df_enc['stroke']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train and Test shapes:", X_train.shape, X_test.shape)
print("Stroke class averages:", y_train.mean().round(4), y_test.mean().round(4))

Train and Test shapes: (4087, 15) (1022, 15)
Stroke class averages: 0.0487 0.0489


In [16]:
bins = [0, 18, 40, 60, 120]
labels = ['0-17', '18-39', '40-59', '60+']
for d in (X_train, X_test):
    d['age_band'] = pd.cut(d['age'], bins=bins, labels=labels, right=False)

band_median = X_train.groupby('age_band', observed=True)['bmi'].median()
global_median = X_train['bmi'].median()

for d in (X_train, X_test):
    fill = d['age_band'].astype(object).map(band_median).astype(float).fillna(global_median)
    d['bmi'] = d['bmi'].fillna(fill)
    d.drop(columns='age_band', inplace=True)

print(band_median)
print(X_train['bmi'].isnull().sum(), X_test['bmi'].isnull().sum())

age_band
0-17     19.9
18-39    27.7
40-59    30.4
60+      29.4
Name: bmi, dtype: float64
0 0


Scale numeric features

In [17]:
from sklearn.preprocessing import StandardScaler

num_cols = ['age', 'avg_glucose_level', 'bmi']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(X_train[num_cols].describe().loc[['mean', 'std']].round(3))

      age  avg_glucose_level  bmi
mean -0.0                0.0  0.0
std   1.0                1.0  1.0


Save the prepared data

In [18]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

P5
Training baseline

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()

baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

Baseline results

In [21]:
y_pred = baseline.predict(X_test)
y_prob = baseline.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 3))

[[972   0]
 [ 49   1]]
              precision    recall  f1-score   support

           0       0.95      1.00      0.98       972
           1       1.00      0.02      0.04        50

    accuracy                           0.95      1022
   macro avg       0.98      0.51      0.51      1022
weighted avg       0.95      0.95      0.93      1022

ROC-AUC: 0.839


 M3 Testing and refining the AI solution

class weighting

In [22]:
balanced = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
balanced.fit(X_train, y_train)

prob_bal = balanced.predict_proba(X_test)[:, 1]
pred_bal = (prob_bal >= 0.5).astype(int)
print(confusion_matrix(y_test, pred_bal))
print(classification_report(y_test, pred_bal, zero_division=0))
print('ROC-AUC:', round(roc_auc_score(y_test, prob_bal), 3))

[[714 258]
 [ 10  40]]
              precision    recall  f1-score   support

           0       0.99      0.73      0.84       972
           1       0.13      0.80      0.23        50

    accuracy                           0.74      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.74      0.81      1022

ROC-AUC: 0.839


Refinement 2: decision-threshold search

In [23]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = cross_val_predict(balanced, X_train, y_train, cv=cv,
                        method='predict_proba')[:, 1]

rows = []
for t in np.linspace(0.01, 0.99, 99):
    p = (oof >= t).astype(int)
    rows.append((t,
                 precision_score(y_train, p, zero_division=0),
                 recall_score(y_train, p)))
search = pd.DataFrame(rows, columns=['threshold', 'precision', 'recall'])

ok = search[search['recall'] >= 0.70]
if ok.empty:
    best_t = search.loc[search['recall'].idxmax(), 'threshold']
else:
    best_t = ok.loc[ok['precision'].idxmax(), 'threshold']
print('Chosen threshold:', round(best_t, 2))

# One evaluation on the untouched test set
pred_tuned = (prob_bal >= best_t).astype(int)
print(confusion_matrix(y_test, pred_tuned))
print(classification_report(y_test, pred_tuned, zero_division=0))

Chosen threshold: 0.58
[[770 202]
 [ 10  40]]
              precision    recall  f1-score   support

           0       0.99      0.79      0.88       972
           1       0.17      0.80      0.27        50

    accuracy                           0.79      1022
   macro avg       0.58      0.80      0.58      1022
weighted avg       0.95      0.79      0.85      1022



Odds ratios

In [24]:
coef = pd.Series(balanced.coef_[0], index=X_train.columns)
odds = np.exp(coef).sort_values(ascending=False).round(3)
print(odds)

age                               6.586
work_type_children                2.455
hypertension                      1.868
smoking_status_smokes             1.331
avg_glucose_level                 1.233
Residence_type                    1.189
heart_disease                     1.153
gender                            1.144
smoking_status_formerly smoked    1.094
bmi                               1.079
work_type_Private                 1.024
work_type_Never_worked            0.923
ever_married                      0.843
work_type_Self-employed           0.813
smoking_status_never smoked       0.791
dtype: float64


Sampling uncertainty

In [25]:
rng = np.random.default_rng(42)
rec, prec = [], []
y_arr = y_test.to_numpy()
for _ in range(1000):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    rec.append(recall_score(y_arr[idx], pred_tuned[idx]))
    prec.append(precision_score(y_arr[idx], pred_tuned[idx], zero_division=0))
print('Recall 95% CI:', np.percentile(rec, [2.5, 97.5]).round(3))
print('Precision 95% CI:', np.percentile(prec, [2.5, 97.5]).round(3))

Recall 95% CI: [0.686 0.911]
Precision 95% CI: [0.12  0.211]
